---
title: Relative-strength new high before price
execute:
  enabled: true
---

`q.indicator.rsnhbp` flags dates when relative strength reaches a new trailing high but the asset price does not. Current observations are compared with highs formed from prior dates only.

This example computes AAPL relative strength against SPY and marks 60-session relative-strength breakouts that precede an AAPL price breakout.

In [1]:
import pandas as pd

import qrt as q

prices = pd.concat(
    {
        "AAPL": q.data.datasets.load("aapl")["close"],
        "SPY": q.data.datasets.load("spy")["close"],
    },
    axis=1,
    join="inner",
).dropna()
end = prices.index.max()
prices = prices.loc[end - pd.DateOffset(years=5) :]
strength = q.indicator.relative_strength(prices["AAPL"], prices["SPY"], lookback=63)
strength.tail()

datetime
2026-07-13    0.113860
2026-07-14    0.117231
2026-07-15    0.176773
2026-07-16    0.176675
2026-07-17    0.206154
Name: relative_strength, dtype: float64

## Calculate the indicator

Pass the asset price and its already-computed relative-strength series. The boolean output preserves the price index.

In [2]:
window = 60
signal = q.indicator.rsnhbp(prices["AAPL"], strength, window=window)
signal.value_counts()

rs_new_high_before_price
False    1196
True       59
Name: count, dtype: int64

## Compare with SPY

AAPL and SPY are indexed to 100 for comparison. Diamond markers identify dates where AAPL's SPY-relative strength made a new high before AAPL's own price did.

In [3]:
indexed = prices.div(prices.iloc[0]).mul(100).add_suffix(" close")
comparison = indexed.assign(**{"AAPL RS high before price": indexed["AAPL close"].where(signal)})
fig = q.plot.col(
    comparison,
    title="AAPL relative-strength highs before price highs, with SPY",
    ylabel="Indexed value (start = 100)",
)
fig.data[-1].update(mode="markers", marker={"size": 7, "symbol": "diamond"})
fig.show(renderer="notebook_connected")